# 📧 Spam Email Detection using BERT

## 🚀 Project Overview

This project implements a **Spam Email Detection System** using **BERT (Bidirectional Encoder Representations from Transformers)**. The model is fine-tuned to classify email or SMS messages into two categories:

* **Spam**
* **Ham (Not Spam)**

The system leverages Natural Language Processing (NLP) and Deep Learning techniques to achieve high classification accuracy and provide reliable spam detection.

---

## 🎯 Objectives

* Detect spam messages automatically.
* Fine-tune a pretrained BERT model for binary text classification.
* Evaluate model performance using multiple metrics.
* Save and deploy the trained model for real-world usage.
* Provide an interactive user interface using Gradio.

---

## 🛠️ Technologies Used

| Technology                | Purpose                        |
| ------------------------- | ------------------------------ |
| Python                    | Programming Language           |
| Pandas                    | Data Processing                |
| NumPy                     | Numerical Operations           |
| PyTorch                   | Deep Learning Framework        |
| Hugging Face Transformers | BERT Model                     |
| Scikit-learn              | Data Splitting & Evaluation    |
| Matplotlib                | Visualization                  |
| Seaborn                   | Confusion Matrix Visualization |
| Gradio                    | Web Interface                  |

---

## 📂 Dataset

The dataset contains labeled messages with two classes:

| Label | Meaning                                    |
| ----- | ------------------------------------------ |
| Spam  | Unwanted promotional or fraudulent message |
| Ham   | Legitimate message                         |

### Sample Data

| Label | Message                                 |
| ----- | --------------------------------------- |
| Spam  | Congratulations! You won a free iPhone. |
| Ham   | Meeting is scheduled at 10 AM tomorrow. |

---

## 🔄 Project Workflow

### 1. Data Loading

* Load dataset using Pandas.
* Explore data structure and class distribution.

### 2. Data Preprocessing

* Remove unnecessary columns.
* Handle missing values.
* Encode labels:

  * Ham → 0
  * Spam → 1

### 3. Train-Test Split

* Split dataset into:

  * Training Set (80%)
  * Validation Set (20%)

### 4. Tokenization

* Use `BertTokenizer` from Hugging Face.
* Convert text into:

  * Input IDs
  * Attention Masks

### 5. Dataset Creation

* Create custom PyTorch Dataset class.
* Use DataLoader for batch processing.

### 6. Model Training

* Load pretrained:

  * `bert-base-uncased`
* Fine-tune for binary classification.
* Optimizer:

  * AdamW
* Learning Rate:

  * 2e-5
* Epochs:

  * 3

### 7. Evaluation

Evaluate model using:

* Accuracy
* Precision
* Recall
* F1 Score
* Confusion Matrix

### 8. Model Saving

Save trained model and tokenizer using:

```python
model.save_pretrained("./spam_bert_model")
tokenizer.save_pretrained("./spam_bert_model")
```

### 9. Deployment

Create a Gradio interface for real-time spam prediction.

---

## 🧠 Model Architecture

Input Email Text

↓

BERT Tokenizer

↓

BERT Base Uncased

↓

Dropout Layer

↓

Classification Head

↓

Spam / Ham Prediction

---

## 📊 Results

### Training Loss

| Epoch | Loss   |
| ----- | ------ |
| 1     | 0.0737 |
| 2     | 0.0173 |
| 3     | 0.0122 |

The model achieved excellent performance with a significant reduction in training loss across epochs.

---

## 📈 Evaluation Metrics

* Accuracy Score
* Precision
* Recall
* F1 Score
* Classification Report
* Confusion Matrix

These metrics help measure the effectiveness of spam detection and identify classification errors.

---

## 🔍 Sample Predictions

### Example 1

**Input:**

```
Congratulations! You have won a $1000 gift card. Click here now.
```

**Prediction:**

```
Spam
```

---

### Example 2

**Input:**

```
Hi John, the meeting has been moved to 3 PM.
```

**Prediction:**

```
Ham
```

---

## ▶️ Running the Project

### Install Dependencies

```bash
pip install transformers torch pandas numpy scikit-learn matplotlib seaborn gradio
```

### Run Training

```bash
python train.py
```

### Launch Gradio App

```bash
python app.py
```

---



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

In [ ]:
df = pd.read_csv('spam.csv', encoding='latin-1')

df = df.dropna(how="any", axis=1)
df.columns = ['label', 'message']

display(df.head())


In [ ]:
print("Dataset Shape:", df.shape)

print("\nClass Distribution:")
print(df['label'].value_counts())

print("\nMissing Values:")
print(df.isnull().sum())

sns.countplot(data=df, x='label')
plt.title('Distribution of Spam vs Ham')
plt.show()

In [ ]:
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

train_text, val_text, train_labels, val_labels = train_test_split(
    df['message'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print("Training Samples:", len(train_text))
print("Validation Samples:", len(val_text))

In [ ]:
tokenizer = BertTokenizer.from_pretrained(
    'bert-base-uncased'
)

train_encodings = tokenizer(
    list(train_text),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_text),
    truncation=True,
    padding=True,
    max_length=128
)

class SpamDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item['labels'] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):
        return len(self.labels)


train_dataset = SpamDataset(
    train_encodings,
    train_labels
)

val_dataset = SpamDataset(
    val_encodings,
    val_labels
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

print("DataLoaders created.")

In [ ]:
batch = next(iter(train_loader))

print("Input IDs Shape:",
      batch['input_ids'].shape)

print("Attention Mask Shape:",
      batch['attention_mask'].shape)

print("Labels Shape:",
      batch['labels'].shape)

In [ ]:
!curl -I https://huggingface.co

In [ ]:
!rm -rf ~/.cache/huggingface

In [ ]:
from huggingface_hub import login

login("")

In [ ]:
device = torch.device(
    'cuda'
) if torch.cuda.is_available() else torch.device('cpu')

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

model.to(device)

print(device)

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
best_accuracy = 0

In [ ]:
from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

# Configuration
epochs = 3
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

train_losses = []
best_f1 = 0

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    print(f"Average Training Loss: {avg_train_loss:.4f}")

    # Validation Phase
    model.eval()
    val_preds = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).flatten()
            val_preds.extend(preds.cpu().numpy())

    # Calculate F1 and save if it's the best so far
    current_f1 = f1_score(val_labels, val_preds)
    print(f"Validation F1 Score: {current_f1:.4f}")

    if current_f1 > best_f1:
        best_f1 = current_f1
        model.save_pretrained("./best_spam_model")
        tokenizer.save_pretrained("./best_spam_model")
        print("--- Best model saved based on F1 Score ---")

print("\nTraining Complete!")

In [ ]:
model.eval()

val_preds = []

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch['input_ids'].to(device)

        attention_mask = batch['attention_mask'].to(device)

        outputs = model(
            input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        preds = torch.argmax(
            logits,
            dim=1
        ).flatten()

        val_preds.extend(
            preds.cpu().numpy()
        )

print(
    classification_report(
        val_labels,
        val_preds,
        target_names=['Ham', 'Spam']
    )
)

cm = confusion_matrix(
    val_labels,
    val_preds
)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.show()

In [ ]:
import pandas as pd

errors_df = pd.DataFrame({
    "Text": list(val_text),
    "Actual": val_labels,
    "Predicted": val_preds
})

errors_df = errors_df[
    errors_df["Actual"] != errors_df["Predicted"]
]

print("Number of Errors:", len(errors_df))

display(errors_df.head(10))

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    val_labels,
    val_preds
)

if accuracy > best_accuracy:

    best_accuracy = accuracy

    model.save_pretrained(
        "./best_spam_model"
    )

    tokenizer.save_pretrained(
        "./best_spam_model"
    )

    print(
        "Best Model Saved!"
    )

In [ ]:
from sklearn.metrics import precision_score

precision = precision_score(
    val_labels,
    val_preds
)

print(
    f"Precision: {precision:.4f}"
)

In [ ]:
from sklearn.metrics import recall_score

recall = recall_score(
    val_labels,
    val_preds
)

print(
    f"Recall: {recall:.4f}"
)

In [ ]:
import os

save_path = "./spam_bert_model"

if not os.path.exists(save_path):
    os.makedirs(save_path)

model.save_pretrained(save_path)

tokenizer.save_pretrained(save_path)

print("Model Saved Successfully")

In [ ]:
from transformers import (
    BertTokenizer,
    BertForSequenceClassification
)

loaded_model = BertForSequenceClassification.from_pretrained(
    "./spam_bert_model"
)

loaded_tokenizer = BertTokenizer.from_pretrained(
    "./spam_bert_model"
)

print("Model Reloaded Successfully!")

In [ ]:
inputs = loaded_tokenizer(
    "Congratulations! You won ₹50000",
    return_tensors="pt"
)

with torch.no_grad():
    outputs = loaded_model(**inputs)

print("Reload Test Successful!")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(
    range(1, len(train_losses)+1),
    train_losses,
    marker='o'
)

plt.title("BERT Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)

plt.show()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

accuracy = accuracy_score(
    val_labels,
    val_preds
)

f1 = f1_score(
    val_labels,
    val_preds
)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
def predict_email(text):

    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():

        outputs = model(**inputs)

        probs = torch.nn.functional.softmax(
            outputs.logits,
            dim=-1
        )

        prediction = torch.argmax(
            probs,
            dim=1
        ).item()

        confidence = probs[0][prediction].item()

    return {
        "Prediction":
            "Spam" if prediction == 1 else "Ham",
        "Confidence":
            f"{confidence*100:.2f}%"
    }

In [ ]:
print(
    predict_email(
        "Congratulations! You have won ₹50000."
    )
)

print(
    predict_email(
        "Hi, are we meeting tomorrow?"
    )
)

In [ ]:
wrong_predictions = []

for text, actual, pred in zip(
    val_text,
    val_labels,
    val_preds
):

    if actual != pred:

        wrong_predictions.append(
            {
                "Text": text,
                "Actual": actual,
                "Predicted": pred
            }
        )

print(
    "Wrong Predictions:",
    len(wrong_predictions)
)

for item in wrong_predictions[:10]:

    print("\nEmail:")
    print(item["Text"])

    print(
        "Actual:",
        item["Actual"]
    )

    print(
        "Predicted:",
        item["Predicted"]
    )

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

def predict_email_gradio(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    model.eval()

    with torch.no_grad():

        outputs = model(**inputs)

        probs = torch.nn.functional.softmax(
            outputs.logits,
            dim=-1
        )

        pred = torch.argmax(
            probs,
            dim=1
        ).item()

        confidence = (
            probs[0][pred].item()
            * 100
        )

    label = (
        "Spam"
        if pred == 1
        else "Ham"
    )

    return (
        f"{label} "
        f"({confidence:.2f}%)"
    )


app = gr.Interface(
    fn=predict_email_gradio,
    inputs="text",
    outputs="text",
    title="Spam Email Detector"
)

app.launch(share=True)

## 🏁 Conclusion and Summary

In this project, we successfully built and fine-tuned a **BERT-based Spam Detection** model.

### Key Achievements:
- **High Performance:** The model reached an accuracy of over **99%** on the validation set.
- **Robustness:** With an F1-score of **0.9695**, the model shows a strong balance between precision and recall, effectively minimizing both false positives and false negatives.
- **Deployment Ready:** We implemented a real-time prediction function and a **Gradio web interface** for easy user interaction.

### Insights from Error Analysis:
By inspecting the 9 incorrect predictions, we noticed that errors often occur in ambiguous cases (e.g., short texts with phone numbers or jokes). Future improvements could involve training on a larger, more diverse dataset or increasing the sequence length during tokenization.